# Melihat dataset kosong

Dari dataset yang uah di jadikan csv, disini mau melihat mana yang empty category dan empty sentiment pada annotator 1, 2, dan 3. serta mengecek apakah isi at, review_ida, dan content setiap baris persis sama di 3 annotator.


In [9]:
import pandas as pd
import os

BASE = r"c:\Users\FATISDA UNS\Documents\Reni\preprocessing\clean_dataset"

df1 = pd.read_csv(os.path.join(BASE, "annotator1_clean.csv"), dtype=str).fillna("")
df2 = pd.read_csv(os.path.join(BASE, "annotator2_clean.csv"), dtype=str).fillna("")
df3 = pd.read_csv(os.path.join(BASE, "annotator3_clean.csv"), dtype=str).fillna("")

print(f"Annotator1: {len(df1)} baris")
print(f"Annotator2: {len(df2)} baris")
print(f"Annotator3: {len(df3)} baris")


Annotator1: 38345 baris
Annotator2: 38345 baris
Annotator3: 38345 baris


## 1. Baris dengan Category atau Sentiment Kosong

In [10]:
def show_empty(df, name):
    empty_cat = df[df["category"] == ""]
    empty_sent = df[df["sentiment"] == ""]
    empty_either = df[(df["category"] == "") | (df["sentiment"] == "")]
    print(f"{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(f"  Empty category  : {len(empty_cat)} baris")
    print(f"  Empty sentiment : {len(empty_sent)} baris")
    print(f"  Empty salah satu/keduanya: {len(empty_either)} baris\n")
    if len(empty_either) > 0:
        display(empty_either[["sentence_id", "at", "content", "category", "sentiment"]].reset_index(drop=True))
    print()

show_empty(df1, "Annotator 1")
show_empty(df2, "Annotator 2")
show_empty(df3, "Annotator 3")


  Annotator 1
  Empty category  : 0 baris
  Empty sentiment : 0 baris
  Empty salah satu/keduanya: 0 baris


  Annotator 2
  Empty category  : 0 baris
  Empty sentiment : 0 baris
  Empty salah satu/keduanya: 0 baris


  Annotator 3
  Empty category  : 18347 baris
  Empty sentiment : 18347 baris
  Empty salah satu/keduanya: 18347 baris



,sentence_id,at,content,category,sentiment
0,review_13324_1,2025-07-14 02:42:07,Perjalanan enak dan nyaman,,
1,review_13325_1,2025-07-14 02:35:32,Alhamdulillah memudahkan belanja tnp keluar ru...,,
2,review_13326_1,2025-07-14 02:27:40,ramah baii,,
3,review_13327_1,2025-07-14 02:11:25,sangat baik dan ramah,,
4,review_13328_1,2025-07-14 01:58:57,terima kasih,,
...,...,...,...,...,...
18342,review_25740_1,2025-05-01 01:11:41,trimakasih bnyk sngat membantu sekali jadi nga...,,
18343,review_25741_1,2025-05-01 01:07:55,ok,,
18344,review_25742_1,2025-05-01 00:21:23,mudah digunakan,,
18345,review_25743_1,2025-05-01 00:21:07,Baru kali ini isi saldo gopay ga masuk pdhal l...,,


## 2. Cek Konsistensi: `sentence_id`, `at`, dan `content` di 3 Annotator

Memastikan setiap baris pada posisi yang sama (row ke-N) memiliki nilai `sentence_id`, `at`, dan `content` yang **persis sama** di ketiga annotator.

In [11]:
cols_check = ["sentence_id", "at", "content"]

# Bandingkan kolom row-by-row antara annotator1 vs annotator2
diff_12 = df1[cols_check].compare(df2[cols_check], result_names=("ann1", "ann2"))
# Bandingkan antara annotator1 vs annotator3
diff_13 = df1[cols_check].compare(df3[cols_check], result_names=("ann1", "ann3"))
# Bandingkan antara annotator2 vs annotator3
diff_23 = df2[cols_check].compare(df3[cols_check], result_names=("ann2", "ann3"))

print(f"Perbedaan Ann1 vs Ann2 : {len(diff_12)} baris berbeda")
print(f"Perbedaan Ann1 vs Ann3 : {len(diff_13)} baris berbeda")
print(f"Perbedaan Ann2 vs Ann3 : {len(diff_23)} baris berbeda")

if len(diff_12) == 0 and len(diff_13) == 0 and len(diff_23) == 0:
    print("\n✅ sentence_id, at, dan content persis sama di ketiga annotator.")
else:
    print("\n⚠️  Ada perbedaan! Detail di bawah:")


Perbedaan Ann1 vs Ann2 : 0 baris berbeda
Perbedaan Ann1 vs Ann3 : 0 baris berbeda
Perbedaan Ann2 vs Ann3 : 0 baris berbeda

✅ sentence_id, at, dan content persis sama di ketiga annotator.


In [ ]:
# Tampilkan detail perbedaan (jika ada)
for label, diff, dfa, dfb in [
    ("Ann1 vs Ann2", diff_12, df1, df2),
    ("Ann1 vs Ann3", diff_13, df1, df3),
    ("Ann2 vs Ann3", diff_23, df2, df3),
]:
    if len(diff) > 0:
        print(f"\n{'='*50}")
        print(f"  Perbedaan {label} ({len(diff)} baris)")
        print(f"{'='*50}")
        idx = diff.index.tolist()
        combined = dfa.loc[idx, cols_check].copy()
        combined.columns = [f"{c}_A" for c in cols_check]
        for c in cols_check:
            combined[f"{c}_B"] = dfb.loc[idx, c].values
        display(combined.reset_index(drop=True))


: 